# CLI Tutorial 1: Quick Start

This tutorial introduces a minimal `treem` command-line workflow. The notebook uses Python only to set up paths, run terminal commands, print command output, and display generated images. The actual morphology operations are all done with the `swc` CLI.

## Environment

Set the repository root, switch to the `tutorials/` directory, create local working folders, and define a helper for running `swc` commands.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

from IPython.display import Image, display


ROOT = Path(
    subprocess.run(
        "git rev-parse --show-toplevel",
        shell=True,
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
).resolve()

os.chdir(ROOT / "tutorials")
assert Path.cwd().resolve() == (ROOT / "tutorials").resolve()

Path("data").mkdir(exist_ok=True)
Path("output").mkdir(exist_ok=True)

f = ROOT / "tests" / "data" / "pass_nmo_1.swc"
assert f.exists(), f"Input file not found: {f}"


def quote(value):
    return '"' + str(value).replace('"', '\\"') + '"'


def run_swc(command, image=None, check=False):
    print(f"$ {command}")
    result = subprocess.run(command, shell=True, capture_output=True, text=True)

    print("\nstdout:")
    print(result.stdout.rstrip() or "(empty)")

    print("\nstderr:")
    print(result.stderr.rstrip() or "(empty)")

    print(f"\nreturn code: {result.returncode}")

    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with return code {result.returncode}: {command}")

    if image is not None:
        image_path = Path(image)
        if image_path.exists():
            display(Image(filename=str(image_path)))
        else:
            print(f"Image was not generated: {image_path}")

    return result


print(f"ROOT = {ROOT}")
print(f"working directory = {Path.cwd()}")
print(f"f = {f}")


## Input File

Start with a sample SWC reconstruction from the repository test data. Check whether it is structurally consistent. If the check succeeds, copy it to `./data/inp.swc`; otherwise, convert it to a compliant SWC file.

In [ ]:
check_result = run_swc(f"swc check {quote(f)}")

inp = Path("data") / "inp.swc"
if check_result.returncode == 0:
    shutil.copy2(f, inp)
    print(f"Copied {f} to {inp}")
else:
    run_swc(f"swc convert {quote(f)} -o ./data/inp.swc -q", check=True)


## View

Visualize the input morphology and save the plot so it can be displayed inside the notebook.

In [ ]:
run_swc("swc view ./data/inp.swc -o ./output/inp.png", image="output/inp.png", check=True)


## Measure

Measure morphometric features of the working input file.

In [ ]:
run_swc("swc measure ./data/inp.swc", check=True)


## Modify

Scale the morphology by a factor of `2` along the x, y, and z axes, and write the modified reconstruction to `./output/scaled.swc`.

In [ ]:
run_swc("swc modify ./data/inp.swc -s 2 2 2 -o ./output/scaled.swc", check=True)


## Compare

View and measure the original working copy and the scaled reconstruction together.

In [ ]:
run_swc(
    "swc view ./data/inp.swc ./output/scaled.swc -c cells -o ./output/compare.png",
    image="output/compare.png",
    check=True,
)


In [ ]:
run_swc("swc measure ./data/inp.swc ./output/scaled.swc", check=True)
